# Master Brain API Bootstrap (Portable)

Run this notebook on a new device to set up and run the API.

What this notebook does:
1. Finds the project root
2. Creates `.env` (if missing) with placeholders
3. Creates `.venv` and installs project dependencies
4. Optionally builds/refreshes the Master Brain index
5. Starts API locally and validates health/auth
6. Optionally installs/updates the Windows auto-start service

> **Security:** do not hardcode real keys in this notebook. Put secrets in `.env` or environment variables.

In [ ]:
from __future__ import annotations

import os
import sys
import time
import json
import ctypes
import platform
import subprocess
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Could not locate project root (missing pyproject.toml/src).")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"PYTHON={sys.executable}")
print(f"OS={platform.system()} {platform.release()}")

In [ ]:
def run(cmd, cwd=None, check=True):
    if isinstance(cmd, list):
        printable = " ".join(str(x) for x in cmd)
    else:
        printable = str(cmd)
    print(f"\n$ {printable}")
    proc = subprocess.run(
        cmd,
        cwd=cwd or PROJECT_ROOT,
        text=True,
        capture_output=True,
        shell=isinstance(cmd, str),
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {printable}")
    return proc

def is_windows_admin() -> bool:
    if platform.system().lower() != "windows":
        return False
    try:
        return bool(ctypes.windll.shell32.IsUserAnAdmin())
    except Exception:
        return False

In [ ]:
env_path = PROJECT_ROOT / '.env'
if not env_path.exists():
    env_path.write_text(
        """OPENAI_API_KEY=your_openai_api_key_here
PERPLEXITY_API_KEY=your_perplexity_api_key_here
OPENAI_EMBED_MODEL=text-embedding-3-small
CLOUD_RERANK_ENABLED=1
BRIDGE_API_KEY=master-brain-bridge-local
BRIDGE_HOST=127.0.0.1
BRIDGE_PORT=8787
BRIDGE_WORKSPACE_ROOT=
BRIDGE_DEFAULT_INDEX_PATH=data/master_brain_index.pkl
MASTER_BRAIN_ROOT=
DISCOGS_TOKEN=
DROPBOX_ACCESS_TOKEN=
""",
        encoding='utf-8'
    )
    print(f"Created {env_path}")
else:
    print(f"Using existing {env_path}")

# Load .env values into this notebook process (without external deps)
for raw in env_path.read_text(encoding='utf-8').splitlines():
    line = raw.strip()
    if not line or line.startswith('#') or '=' not in line:
        continue
    k, v = line.split('=', 1)
    os.environ[k.strip()] = v.strip()

print('Tip: set BRIDGE_API_KEY in .env if you want a custom key. Default fallback is master-brain-bridge-local.')

In [ ]:
venv_python = PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe' if platform.system().lower() == 'windows' else PROJECT_ROOT / '.venv' / 'bin' / 'python'

if not venv_python.exists():
    run([sys.executable, '-m', 'venv', str(PROJECT_ROOT / '.venv')])

run([str(venv_python), '-m', 'pip', 'install', '--upgrade', 'pip'])
run([str(venv_python), '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)])
print(f"Ready interpreter: {venv_python}")

In [ ]:
RUN_INDEX_BUILD = False  # set True if you want to (re)build index now
MASTER_ROOT = os.environ.get('MASTER_BRAIN_ROOT') or str(PROJECT_ROOT / 'Master Brain')

if RUN_INDEX_BUILD:
    run([
        str(venv_python), '-m', 'math_logic_agent.cli', 'build-master-brain',
        '--master-root', MASTER_ROOT,
        '--module-config', 'config/master_brain.toml',
        '--index-path', 'data/master_brain_index.pkl',
        '--incremental',
        '--quarantine-path', 'data/master_brain_quarantine.json',
        '--checkpoint-path', 'data/master_brain_checkpoint.json',
        '--checkpoint-every', '200',
        '--respect-quarantine',
    ])
else:
    print('Skipping index build. Set RUN_INDEX_BUILD=True to run it.')

In [ ]:
RUN_LOCAL_API = True
API_HOST = os.environ.get('BRIDGE_HOST', '127.0.0.1')
API_PORT = int(os.environ.get('BRIDGE_PORT', '8787'))
pid_file = PROJECT_ROOT / '.api_pid'

def api_alive(host: str, port: int) -> bool:
    import urllib.request
    try:
        with urllib.request.urlopen(f'http://{host}:{port}/health', timeout=2) as r:
            return int(r.status) == 200
    except Exception:
        return False

if RUN_LOCAL_API:
    if api_alive(API_HOST, API_PORT):
        print(f"API already running at http://{API_HOST}:{API_PORT}; skipping new local process.")
    else:
        proc = subprocess.Popen(
            [str(venv_python), '-m', 'uvicorn', 'math_logic_agent.api:app', '--host', API_HOST, '--port', str(API_PORT)],
            cwd=PROJECT_ROOT,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        pid_file.write_text(str(proc.pid), encoding='utf-8')
        print(f"Started local API process pid={proc.pid}")
else:
    print('Skipping local API start.')

In [ ]:
import urllib.request
import urllib.error
import json

API_HOST = os.environ.get('BRIDGE_HOST', '127.0.0.1')
API_PORT = int(os.environ.get('BRIDGE_PORT', '8787'))
base = f"http://{API_HOST}:{API_PORT}"

def mask_secret(value: str | None, left: int = 4, right: int = 4) -> str:
    if not value:
        return '<missing>'
    v = str(value).strip()
    if len(v) <= left + right:
        return '*' * len(v)
    return f"{v[:left]}...{v[-right:]}"

def http_get(url: str, headers: dict | None = None, timeout: int = 8):
    req = urllib.request.Request(url, headers=headers or {}, method='GET')
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            status = int(resp.status)
            body = resp.read().decode('utf-8', errors='replace')
            return status, body
    except urllib.error.HTTPError as e:
        body = e.read().decode('utf-8', errors='replace') if getattr(e, 'fp', None) else str(e)
        return int(e.code), body

health_status = None
health_body = ''
for _ in range(20):
    try:
        health_status, health_body = http_get(f"{base}/health", timeout=3)
        if health_status == 200:
            break
    except Exception:
        pass
    time.sleep(1)

if health_status is None:
    raise RuntimeError(f"API did not respond at {base}/health after retries.")

print('Health:', health_status, health_body[:300])

bridge_key = os.environ.get('BRIDGE_API_KEY', 'master-brain-bridge-local')
cfg_status, cfg_body = http_get(f"{base}/v1/config", headers={'x-api-key': bridge_key}, timeout=8)
print('Config status:', cfg_status)

if cfg_status == 200:
    try:
        cfg_json = json.loads(cfg_body)
        redacted = {}
        for k, v in cfg_json.items():
            if 'key' in k.lower() or 'token' in k.lower():
                redacted[k] = mask_secret(v)
            else:
                redacted[k] = v
        print(json.dumps(redacted, indent=2)[:1200])
    except Exception:
        print(cfg_body[:500])
else:
    print(cfg_body[:500])

In [ ]:
# Auth diagnostics (safe): helps explain 401 without printing full secrets
from pathlib import Path
import urllib.request
import urllib.error

def mask_secret(value: str | None, left: int = 4, right: int = 4) -> str:
    if not value:
        return '<missing>'
    v = value.strip()
    if len(v) <= left + right:
        return '*' * len(v)
    return f"{v[:left]}...{v[-right:]}"

def http_status(url: str, headers: dict | None = None, timeout: int = 10) -> int:
    req = urllib.request.Request(url, headers=headers or {}, method='GET')
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return int(resp.status)
    except urllib.error.HTTPError as e:
        return int(e.code)

env_file = PROJECT_ROOT / '.env'
env_key_file = None
if env_file.exists():
    for line in env_file.read_text(encoding='utf-8').splitlines():
        if line.strip().startswith('BRIDGE_API_KEY='):
            env_key_file = line.split('=', 1)[1].strip()
            break

env_key_runtime = os.environ.get('BRIDGE_API_KEY')
candidate = bridge_key

print('Notebook key candidates (masked):')
print(' - from .env file :', mask_secret(env_key_file))
print(' - from os.environ:', mask_secret(env_key_runtime))
print(' - in notebook var:', mask_secret(candidate))

r_x = http_status(f"{base}/v1/config", headers={'x-api-key': candidate}, timeout=10)
r_b = http_status(f"{base}/v1/config", headers={'Authorization': f'Bearer {candidate}'}, timeout=10)

print('Auth probe status codes:')
print(' - x-api-key header      ->', r_x)
print(' - Authorization Bearer  ->', r_b)

if r_x == 401 and r_b == 401:
    print('Diagnosis: key mismatch between notebook and running service, or service not reloaded.')
elif r_x == 200 and r_b == 401:
    print('Diagnosis: service currently expects x-api-key only (older auth logic).')
elif r_b == 200 and r_x == 401:
    print('Diagnosis: service currently expects bearer auth path for this client.')
else:
    print('Diagnosis: auth looks healthy for at least one header format.')

In [ ]:
# Verify Dropbox token wiring through the bridge API
import urllib.request
import urllib.error

bridge_key = os.environ.get('BRIDGE_API_KEY', 'master-brain-bridge-local')
req = urllib.request.Request(f"{base}/v1/dropbox-health", headers={'x-api-key': bridge_key}, method='GET')
try:
    with urllib.request.urlopen(req, timeout=12) as resp:
        drop_status = int(resp.status)
        drop_body = resp.read().decode('utf-8', errors='replace')
except urllib.error.HTTPError as e:
    drop_status = int(e.code)
    drop_body = e.read().decode('utf-8', errors='replace') if getattr(e, 'fp', None) else str(e)

print('Dropbox health status:', drop_status)
print(drop_body[:1200])

In [ ]:
INSTALL_WINDOWS_SERVICE = False  # set True to install/update auto-start service

if platform.system().lower() == 'windows' and INSTALL_WINDOWS_SERVICE:
    if not is_windows_admin():
        raise PermissionError('Run Jupyter/VS Code as Administrator before enabling INSTALL_WINDOWS_SERVICE=True')
    run('powershell -NoProfile -ExecutionPolicy Bypass -File scripts/install-masterbrain-service.ps1')
    run('sc.exe query MasterBrainBridgeAPI')
else:
    print('Skipping service install. Set INSTALL_WINDOWS_SERVICE=True (and run as admin) to enable persistent startup.')

## Notes

- For **portable use on another device**: clone repo, open this notebook, run cells top-to-bottom.
- For **persistent background API on Windows startup**: enable the service-install cell and run notebook with Administrator privileges.
- Default notebook/API key if not overridden: `master-brain-bridge-local`.
- You can stop local API process from this notebook by killing PID in `.api_pid` or by using the stop service script if running as service.